### Insert here title

1. Motorcycles per capita (-0.357): Higher motorcycle ownership suggests middle-class areas with better transportation access. In Brazil, motorcycles are often a first vehicle purchase for working-class families, indicating areas where people have means to get to appointments.

2. Altitude (0.279): Higher altitude areas in Brazil often correspond to more remote or mountainous regions with less developed infrastructure and harder access to healthcare facilities.

3. Post offices per capita (-0.217): In Brazil, post office density indicates urban development and government service presence. Better-served areas show lower no-show rates.

4. Fixed phones per capita (0.204): Interestingly positive correlation. May indicate older, possibly retired populations who have landlines but might have mobility issues.

5. Area (-0.125): Larger areas typically mean more rural settings in Brazil. Negative correlation suggests rural patients, once they commit to appointments, are more likely to show up.

6. IDHM metrics (small correlations): The Human Development Index components show weak correlations, suggesting socioeconomic factors might be less important than infrastructure and accessibility.

7. Healthcare companies per capita (0.055): Minimal correlation suggests availability of healthcare facilities isn't a major factor in no-shows

### 1.Imports

In [ ]:
# !pip install plotly
# !pip install nbformat
# !pip install geopandas contextily
# !pip install osmnx

import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox

df_appointments = pd.read_csv('healthcare-noshow/healthcare_noshows_appt.csv')
df_cities = pd.read_csv('cities/BRAZIL_CITIES_REV2022.csv')

display(df_appointments.head())
display(df_appointments.info())
display(df_appointments.describe())
display(df_appointments.shape)

print("\nCities Dataset:\n")
display(df_cities.head())
display(df_cities.info())
display(df_cities.describe())
display(df_cities.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'noshow/healthcare_noshows_appt.csv'

In [ ]:
# Standardize both datasets
df_appointments['City_Name'] = df_appointments['Neighbourhood'].str.title()
df_cities['City_Std'] = df_cities['CITY'].str.upper()

# Check duplicates in cities
city_duplicates = df_cities[df_cities.duplicated(subset=['CITY'], keep=False)]
print("Duplicate city counts:")
print(city_duplicates['CITY'].value_counts())

# Which state(s) is your appointments data from?
print("\nUnique neighborhoods:")
print(df_appointments['Neighbourhood'].nunique())

In [ ]:
appointment_cities = df_appointments['Neighbourhood'].unique()
appointment_cities_title = [city.title() for city in appointment_cities]
mapped_cities = df_cities[df_cities['CITY'].isin(appointment_cities_title)]

plt.figure(figsize=(15,10))
for state in mapped_cities['STATE'].unique():
    state_data = mapped_cities[mapped_cities['STATE'] == state]
    plt.scatter(state_data['LONG'], state_data['LAT'], label=state, alpha=0.6)

plt.legend()
plt.title('Healthcare Appointment Cities by State')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

print(f"Cities found: {len(mapped_cities)}")
print("\nCities by state:")
print(mapped_cities['STATE'].value_counts())

In [ ]:
# Get Brazil geometry and prepare data
brazil = ox.geocode_to_gdf('Brazil')
city_duplicates = df_cities[df_cities.duplicated(subset=['CITY'], keep=False)]

# Count appointments per neighborhood
appointment_counts = df_appointments['Neighbourhood'].value_counts()

# Prepare mapped cities with appointment counts
mapped_cities = df_cities[df_cities['CITY'].isin([c.title() for c in appointment_counts.index])]
mapped_cities['appointment_count'] = mapped_cities['CITY'].apply(
   lambda x: appointment_counts.get(x.upper(), 0)
)
mapped_cities['duplicate_group'] = mapped_cities['CITY'].map(
   {city: idx for idx, city in enumerate(city_duplicates['CITY'].unique())}
)

# Markers for different duplicate groups
markers = ['*', 'P', 'D', 'X', '^', '<', '>', 'v', 's', 'p']

# Create plot
fig, ax = plt.subplots(figsize=(15, 10))
brazil.plot(ax=ax, alpha=0.5, color='lightgray')

# Plot cities
for state in mapped_cities['STATE'].unique():
   state_data = mapped_cities[mapped_cities['STATE'] == state]
   
   # Non-duplicates
   non_dup = state_data[state_data['duplicate_group'].isna()]
   plt.scatter(non_dup['LONG'], non_dup['LAT'], 
              s=non_dup['appointment_count']/100,  # Size based on appointments
              label=f"{state}", alpha=0.6)
   
   # Duplicates with different markers
   for group in state_data['duplicate_group'].dropna().unique():
       dup = state_data[state_data['duplicate_group'] == group]
       marker = markers[int(group % len(markers))]
       plt.scatter(dup['LONG'], dup['LAT'], 
                  s=dup['appointment_count']/100,  # Size based on appointments
                  marker=marker, 
                  label=f"{dup['CITY'].iloc[0]} ({state})", 
                  alpha=0.8)

# Add hover annotations
annot = ax.annotate("", xy=(0,0), xytext=(10,10), textcoords="offset points",
                   bbox=dict(boxstyle="round", fc="w", alpha=0.8))
annot.set_visible(False)

def hover(event):
   if event.inaxes == ax:
       for collection in ax.collections:
           cont, ind = collection.contains(event)
           if cont:
               pos = collection.get_offsets()[ind["ind"][0]]
               city_data = mapped_cities[(mapped_cities['LONG'] == pos[0]) & 
                                      (mapped_cities['LAT'] == pos[1])].iloc[0]
               text = (f"City: {city_data['CITY']}\n"
                      f"State: {city_data['STATE']}\n"
                      f"Appointments: {city_data['appointment_count']:,}")
               if not np.isnan(city_data.get('duplicate_group', np.nan)):
                   text += f"\nPart of duplicate group {int(city_data['duplicate_group'])}"
               annot.xy = pos
               annot.set_text(text)
               annot.set_visible(True)
               fig.canvas.draw_idle()
               return
       annot.set_visible(False)
       fig.canvas.draw_idle()

fig.canvas.mpl_connect("motion_notify_event", hover)

plt.title('Healthcare Appointments by City\nMarker size indicates number of appointments')
ax.axis('off')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Get Brazil map
brazil = ox.geocode_to_gdf('Brazil')

# Get unique neighborhoods and count appointments
unique_neighborhoods = df_appointments['Neighbourhood'].unique()
appointment_counts = df_appointments['Neighbourhood'].value_counts()

# Create mapping DataFrame
mapped_cities = df_cities[df_cities['CITY'].isin([n.title() for n in unique_neighborhoods])]
mapped_cities['appointment_count'] = mapped_cities['CITY'].apply(
   lambda x: appointment_counts.get(x.upper(), 0)
)

# Plot
fig, ax = plt.subplots(figsize=(15, 10))
brazil.plot(ax=ax, alpha=0.5, color='lightgray')

for state in mapped_cities['STATE'].unique():
   state_data = mapped_cities[mapped_cities['STATE'] == state]
   plt.scatter(state_data['LONG'], state_data['LAT'], 
              s=state_data['appointment_count']/100,
              label=state, alpha=0.6)

# Hover functionality
annot = ax.annotate("", xy=(0,0), xytext=(10,10), textcoords="offset points",
                   bbox=dict(boxstyle="round", fc="w", alpha=0.8))
annot.set_visible(False)

def hover(event):
   if event.inaxes == ax:
       for collection in ax.collections:
           cont, ind = collection.contains(event)
           if cont:
               pos = collection.get_offsets()[ind["ind"][0]]
               city_data = mapped_cities[(mapped_cities['LONG'] == pos[0]) & 
                                      (mapped_cities['LAT'] == pos[1])].iloc[0]
               text = (f"City: {city_data['CITY']}\n"
                      f"State: {city_data['STATE']}\n"
                      f"Appointments: {city_data['appointment_count']:,}")
               annot.xy = pos
               annot.set_text(text)
               annot.set_visible(True)
               fig.canvas.draw_idle()
               return
       annot.set_visible(False)
       fig.canvas.draw_idle()

fig.canvas.mpl_connect("motion_notify_event", hover)

plt.title('Healthcare Appointments by Unique Cities\nMarker size indicates number of appointments')
ax.axis('off')
plt.legend()
plt.show()

print(f"Unique cities mapped: {len(mapped_cities)}")

In [ ]:
# Select meaningful features, excluding redundant or administrative ones
features_to_analyze = [
   'IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao',
   'GDP_CAPITA', 'GDP', 'GVA_TOTAL', 
   'FIXED_PHONES', 'PAY_TV',
   'Pr_Bank', 'Pu_Bank', 'Pr_Assets', 'Pu_Assets',
   'POST_OFFICES', 'HOTELS', 'BEDS',
   'IBGE_RES_POP', 'ESTIMATED_POP', 'POP_GDP',
   'Cars', 'Motorcycles',
   'AREA', 'ALT',
   'COMP_TOT', 'COMP_Q',  # Total companies and healthcare companies
   'MUN_EXPENDIT'
]

# Calculate no-show rates and merge
no_show_rates = df_appointments.groupby('Neighbourhood').agg({
   'Showed_up': lambda x: (~x).mean()
}).reset_index()
no_show_rates.columns = ['CITY', 'no_show_rate']
no_show_rates['CITY'] = no_show_rates['CITY'].str.title()
city_features = pd.merge(no_show_rates, df_cities, on='CITY', how='left')

# Calculate and display correlations
correlations = {feature: city_features['no_show_rate'].corr(city_features[feature]) 
              for feature in features_to_analyze if city_features[feature].dtype in ['float64', 'int64']}
correlations_sorted = dict(sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True))

print("Top 10 features correlated with no-show rates:")
for feature, corr in list(correlations_sorted.items())[:10]:
   print(f"{feature}: {corr:.3f}")

# Visualize top correlations
plt.figure(figsize=(12, 6))
plt.bar(list(correlations_sorted.keys())[:10], [abs(v) for v in correlations_sorted.values()][:10])
plt.xticks(rotation=45, ha='right')
plt.title('Absolute Correlation with No-show Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate per capita metrics
def add_per_capita(df):
   pop = df['IBGE_RES_POP'].replace(0, np.nan)  # Avoid division by zero
   return df.assign(
       cars_per_capita=df['Cars'] / pop,
       motorcycles_per_capita=df['Motorcycles'] / pop,
       banks_per_capita=(df['Pr_Bank'] + df['Pu_Bank']) / pop,
       healthcare_companies_per_capita=df['COMP_Q'] / pop,
       hotels_per_capita=df['HOTELS'] / pop,
       post_offices_per_capita=df['POST_OFFICES'] / pop,
       phones_per_capita=df['FIXED_PHONES'] / pop,
       paytv_per_capita=df['PAY_TV'] / pop
   )

# Prepare data
no_show_rates = df_appointments.groupby('Neighbourhood').agg({
   'Showed_up': lambda x: (~x).mean()
}).reset_index()
no_show_rates.columns = ['CITY', 'no_show_rate']
no_show_rates['CITY'] = no_show_rates['CITY'].str.title()

# Merge and add per capita
city_features = pd.merge(no_show_rates, df_cities, on='CITY', how='left')
city_features = add_per_capita(city_features)

# Features including per capita
features = [
   'IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao',
   'GDP_CAPITA', 'ALT', 'AREA',
   'cars_per_capita', 'motorcycles_per_capita', 'banks_per_capita',
   'healthcare_companies_per_capita', 'hotels_per_capita',
   'post_offices_per_capita', 'phones_per_capita', 'paytv_per_capita'
]

# Calculate correlations
correlations = {f: city_features['no_show_rate'].corr(city_features[f]) 
              for f in features if city_features[f].dtype in ['float64', 'int64']}
correlations_sorted = dict(sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True))

# Plot
plt.figure(figsize=(12, 6))
plt.bar(list(correlations_sorted.keys())[:10], 
       [abs(v) for v in correlations_sorted.values()][:10])
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Features: Absolute Correlation with No-show Rate')
plt.tight_layout()
plt.show()

print("\nCorrelations (positive means higher no-show rate):")
for f, c in correlations_sorted.items():
   print(f"{f}: {c:.3f}")

Here's an analysis of the key correlations in the context of Brazilian healthcare and poverty:

1. Motorcycles per capita (-0.357): Higher motorcycle ownership suggests middle-class areas with better transportation access. In Brazil, motorcycles are often a first vehicle purchase for working-class families, indicating areas where people have means to get to appointments.

2. Altitude (0.279): Higher altitude areas in Brazil often correspond to more remote or mountainous regions with less developed infrastructure and harder access to healthcare facilities.

3. Post offices per capita (-0.217): In Brazil, post office density indicates urban development and government service presence. Better-served areas show lower no-show rates.

4. Fixed phones per capita (0.204): Interestingly positive correlation. May indicate older, possibly retired populations who have landlines but might have mobility issues.

5. Area (-0.125): Larger areas typically mean more rural settings in Brazil. Negative correlation suggests rural patients, once they commit to appointments, are more likely to show up.

6. IDHM metrics (small correlations): The Human Development Index components show weak correlations, suggesting socioeconomic factors might be less important than infrastructure and accessibility.

7. Healthcare companies per capita (0.055): Minimal correlation suggests availability of healthcare facilities isn't a major factor in no-shows.